# Download & Preprocessing DEM & Slope — Google Earth Engine (GEE)
**Kabupaten Banjarnegara, Jawa Tengah**

Project GEE: `riset-banjarnegara`

Notebook ini menggunakan Google Earth Engine resmi untuk:
1. Mengunduh data **Copernicus DEM 30m** (`COPERNICUS/DEM/GLO30`).
2. Menghitung **Kemiringan Lereng (Slope)** secara cloud-computing di server Google.
3. Melakukan **Reklasifikasi Kelas Kemiringan Lereng** standar PUPR / SK Mentan.
4. Menghasilkan **GeoJSON Vektor Ringan (< 2 MB)** `slope_banjarnegara.geojson` untuk aplikasi WebGIS.

## 1. Install & Import Libraries

In [ ]:
!pip install earthengine-api geemap geopandas osmnx shapely matplotlib -q

In [ ]:
import ee
import geemap
import geopandas as gpd
import osmnx as ox
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries berhasil dimuat.")

## 2. Inisialisasi Google Earth Engine
Menggunakan Project ID: `riset-banjarnegara`

In [ ]:
GEE_PROJECT_ID = "riset-banjarnegara"

# Autentikasi dan Inisialisasi Earth Engine
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"✅ GEE Berhasil diinisialisasi dengan project: {GEE_PROJECT_ID}")
except Exception as e:
    print("Meminta autentikasi GEE...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"✅ GEE Berhasil diinisialisasi dengan project: {GEE_PROJECT_ID}")

## 3. Ambil Batas Wilayah Banjarnegara

In [ ]:
PLACE_NAME = "Kabupaten Banjarnegara, Jawa Tengah, Indonesia"
print(f"Mengambil boundary: {PLACE_NAME}...")
boundary_gdf = ox.geocode_to_gdf(PLACE_NAME)

# Konversi boundary GeoPandas ke ee.Geometry
boundary_json = json.loads(boundary_gdf.to_json())
roi = ee.Geometry(boundary_json['features'][0]['geometry'])

print("Boundary Banjarnegara siap digunakan di GEE.")
boundary_gdf.plot(edgecolor='red', facecolor='none', figsize=(6, 6))
plt.title("Batas Wilayah Kabupaten Banjarnegara")
plt.show()

## 4. Proses DEM & Hitung Kemiringan Lereng (Slope) di GEE Cloud

In [ ]:
# 1. Ambil Copernicus DEM 30m Global dan clip sesuai Banjarnegara
dem = ee.ImageCollection("COPERNICUS/DEM/GLO30") \
        .select('DEM') \
        .mosaic() \
        .clip(roi)

# 2. Hitung Slope dalam derajat via GEE built-in algorithm
slope_deg = ee.Terrain.slope(dem)

# 3. Konversi Slope Derajat ke Persen: slope_pct = tan(deg * pi / 180) * 100
slope_pct = slope_deg.multiply(3.14159265359 / 180).tan().multiply(100)

# 4. Reklasifikasi Kelas Kemiringan Lereng (Standar PUPR / SK Mentan)
# Kelas 1: 0 - 8% (Datar)
# Kelas 2: 8 - 15% (Landai)
# Kelas 3: 15 - 25% (Agak Curam)
# Kelas 4: 25 - 40% (Curam)
# Kelas 5: > 40% (Sangat Curam)
slope_class = ee.Image(1) \
    .where(slope_pct.gt(8).And(slope_pct.lte(15)), 2) \
    .where(slope_pct.gt(15).And(slope_pct.lte(25)), 3) \
    .where(slope_pct.gt(25).And(slope_pct.lte(40)), 4) \
    .where(slope_pct.gt(40), 5) \
    .clip(roi)

print("✅ Perhitungan DEM dan Slope di cloud GEE selesai.")

## 5. Visualisasi Interaktif di Peta

In [ ]:
Map = geemap.Map(center=[-7.35, 109.65], zoom=10)

dem_vis = {'min': 50, 'max': 2500, 'palette': ['blue', 'green', 'yellow', 'orange', 'brown', 'white']}
slope_vis = {'min': 1, 'max': 5, 'palette': ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']}

Map.addLayer(dem, dem_vis, 'Copernicus DEM 30m')
Map.addLayer(slope_class, slope_vis, 'Kelas Kemiringan Lereng')
Map.addLayer(roi, {'color': 'black'}, 'Batas Banjarnegara', False)
Map.add_legend(title='Kelas Kemiringan Lereng', 
               labels=['1. Datar (0-8%)', '2. Landai (8-15%)', '3. Agak Curam (15-25%)', '4. Curam (25-40%)', '5. Sangat Curam (>40%)'],
               colors=['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027'])
Map

## 6. Vektorisasi & Export ke GeoJSON Ringan (< 2 MB)

In [ ]:
print("Mengekstrak polygon vektor dari GEE...")

# Vektorisasi raster slope_class dengan skala 60 meter (ringan dan presisi)
vectors = slope_class.reduceToVectors(
    geometry=roi,
    crs=slope_class.projection(),
    scale=60,
    geometryType='polygon',
    eightConnected=False,
    labelProperty='kelas_slope',
    maxPixels=1e8
)

# Konversi ee.FeatureCollection ke GeoDataFrame lokal
print("Mengonversi ke GeoPandas DataFrame...")
gdf_slope = geemap.ee_to_gdf(vectors)

# Metadata Kelas
CLASS_META = {
    1: {"label": "0 - 8% (Datar)", "score": 100, "color": "#1a9850"},
    2: {"label": "8 - 15% (Landai)", "score": 80, "color": "#91cf60"},
    3: {"label": "15 - 25% (Agak Curam)", "score": 50, "color": "#fee08b"},
    4: {"label": "25 - 40% (Curam)", "score": 20, "color": "#fc8d59"},
    5: {"label": "> 40% (Sangat Curam)", "score": 0, "color": "#d73027"}
}

# Dissolve per kelas lereng agar geometri rapi dan ukuran file kecil
print("Melakukan dissolve geometri per kelas...")
gdf_dissolved = gdf_slope.dissolve(by='kelas_slope').reset_index()

# Tambahkan atribut deskriptif
gdf_dissolved['kategori'] = gdf_dissolved['kelas_slope'].map(lambda k: CLASS_META.get(k, {}).get('label', 'Lainnya'))
gdf_dissolved['skor'] = gdf_dissolved['kelas_slope'].map(lambda k: CLASS_META.get(k, {}).get('score', 0))
gdf_dissolved['warna'] = gdf_dissolved['kelas_slope'].map(lambda k: CLASS_META.get(k, {}).get('color', '#cccccc'))

# Sederhanakan geometri (simplify) toleransi ~30 meter agar file ringan di WebGIS
print("Menyederhanakan geometri...")
gdf_dissolved['geometry'] = gdf_dissolved['geometry'].simplify(tolerance=0.0003, preserve_topology=True)

# Export ke GeoJSON
output_slope_geojson = "slope_banjarnegara.geojson"
gdf_dissolved.to_file(output_slope_geojson, driver="GeoJSON")

file_size_mb = len(open(output_slope_geojson, 'rb').read()) / (1024 * 1024)
print(f"\n✅ GeoJSON Slope Berhasil Dibuat: {output_slope_geojson}")
print(f"   Ukuran File: {file_size_mb:.2f} MB (Ringan & Cepat di WebGIS!)")

## 7. Unduh File ke Komputer

In [ ]:
try:
    from google.colab import files
    print("Mengunduh slope_banjarnegara.geojson...")
    files.download(output_slope_geojson)
    print("📥 Download otomatis dimulai!")
except ImportError:
    print("File tersimpan di:", output_slope_geojson)

## 8. Petunjuk Penyimpanan di Proyek

Simpan file hasil unduhan di folder:

`U:\Project\rekomtps\data\DOWNLOAD\slope_banjarnegara.geojson`